# In Class Activity 04/28 Naive Bayes
**Amrit Dhillon**

**#1.** Create a new notebook for this assignment. [*DONE*]

**#2.** Build a Gaussian Naive Bayes model to predict irrigation need using the data for the Kaggle playground series competition S6E4. You may use your own existing data preprocessing pipeline or adapt from the in-class adult notebook. You should train and test using only the Kaggle train data. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report


In [2]:
id = pd.read_csv("/Users/amritdhillon/Desktop/Advanced ML/naive bayes in class/playground-series-s6e4/train.csv")

X = id.drop(["Irrigation_Need", "id"], axis=1)
y = id["Irrigation_Need"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42)  

#identify numeric/categorical columns and define preprocessing
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

#gaussian naive bayes pipeline, fit, and test evaluation
nb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", FunctionTransformer(to_dense)),
    ("model", GaussianNB())
])

nb_model.fit(X_train, y_train)

baseline_preds = nb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_preds))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, baseline_preds))
print("Classification Report:")
print(classification_report(y_test, baseline_preds, target_names=label_encoder.classes_))


Accuracy: 0.7847460317460317
Balanced Accuracy: 0.7794479358774254

Classification Report:
              precision    recall  f1-score   support

        High       0.53      0.78      0.63      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.70      0.77      0.73     47815

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000



**#3.** Generate predicted probabilities using your model.

In [3]:
nb_probs = nb_model.predict_proba(X_test)

print("Probability matrix shape:", nb_probs.shape)
print("Class order:", label_encoder.classes_)
print("First 5 probability rows:")
print(nb_probs[:5])


Probability matrix shape: (126000, 3)
Class order: ['High' 'Low' 'Medium']
First 5 probability rows:
[[1.78464027e-21 9.98828421e-01 1.17157925e-03]
 [2.33239306e-02 2.12419514e-02 9.55434118e-01]
 [6.50956484e-06 5.41477706e-02 9.45845720e-01]
 [4.26988327e-15 9.96149830e-01 3.85016993e-03]
 [5.24086352e-15 9.65293444e-01 3.47065561e-02]]


**#4.** Create baseline predictions using the default rule (choose the class with the highest probability).

In [6]:
#pick class with highest probability
baseline_pred = np.argmax(nb_probs, axis=1)
baseline_pred_labels = label_encoder.inverse_transform(baseline_pred)

print("First 10 baseline class predictions:")
print(baseline_pred_labels[:10])


First 10 baseline class predictions:
['Low' 'Medium' 'Medium' 'Low' 'Low' 'Low' 'Medium' 'Medium' 'Medium'
 'Medium']


**#5.** Choose one class to focus on (preferably a rare or important class) and select one evaluation metric (e.g., F1, recall, or balanced accuracy).

Chosen Class:

- High (Rare class which didn't appeat in my baseline class predictions, so I want to focus on it)

Chosen Evaluation Metric:

- Recall (Since high is a rare class, recall is a strong choice as missing the classification of a high class is worse than over-predicting it.
I think it's the best choice as it directly measures how many high cases were successfully caught.)

**#6.** Apply a threshold to your chosen class by deciding what probability is “high enough” to assign that class. Generate a second set of predictions using this threshold.


In [7]:
#thresholding for the High class, if P(High) >= threshold, predict High otherwise keep baseline prediction

high = np.where(label_encoder.classes_ == "High")[0][0]
high_threshold = 0.20  #I can change this to whatever to see the differences

threshold_pred = baseline_pred.copy()
threshold_pred[nb_probs[:, high] >= high_threshold] = high

threshold_pred_labels = label_encoder.inverse_transform(threshold_pred)

print("Chosen class:", "High")
print("Threshold:", high_threshold)
print("Classification report (thresholded predictions):")
print(classification_report(y_test, threshold_pred, target_names=label_encoder.classes_))


Chosen class: High
Threshold: 0.2
Classification report (thresholded predictions):
              precision    recall  f1-score   support

        High       0.36      0.83      0.50      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.68      0.70      0.69     47815

    accuracy                           0.76    126000
   macro avg       0.64      0.78      0.68    126000
weighted avg       0.79      0.76      0.77    126000



**#7.** Using a markdown cell, discuss which class you selected, what threshold you chose, how the metric changed, what tradeoff you observed, and how Naive Bayes compares to your existing model(s).  You should use a classification report to assist with this discussion.

- I focused on the High irrigation class because it was the rarest class and likely the most important one to catch correctly because of this

- I used recall for High as my main metric and set a class threshold of 0.20 (predict High when P(High) ≥ 0.20)

- After using this threshold my classification report showed:

    - High recall = 0.83
    - High precision = 0.36
    - High F1 = 0.50
    - Overall accuracy = 0.76

- This threshold rule improved my ability to detect High cases, but the tradeoff is more false positives for High, which lowered precision and reduced overall accuracy, so overall the model became more sensitive to High cases but less conservative in general

- Compared with my existing boosting models (XGBoost/LightGBM/HistGradientBoosting in homeworks), Naive Bayes is clearly weaker overall in predictive performance (ex: XGBoost kaggle score of 0.96830 vs Naive Bayes test accuracy of 0.76), but the threshold rule makes it more useful when the goal is to prioritize catching rare High-irrigation events or events like it
